In [94]:
:clear

In [95]:
:dep pecos = { version = "0.1.1", path = "../crates/pecos" }

In [96]:
// Bell State Example using SparseStab simulator
use pecos::prelude::*;

// Create a 2-qubit stabilizer simulator
let mut sim = SparseStab::new(2);

// Create Bell state: (|00⟩ + |11⟩)/√2
sim.h(0)      // Apply Hadamard to qubit 0: |00⟩ -> (|00⟩ + |10⟩)/√2
   .cx(0, 1); // Apply CNOT: (|00⟩ + |10⟩)/√2 -> (|00⟩ + |11⟩)/√2

// Measure both qubits
let result0: MeasurementResult = sim.mz(0);
let result1: MeasurementResult = sim.mz(1);

println!("Bell State Measurement Results:");
println!("  Qubit 0: {} (deterministic: {})", 
         if result0.outcome { "1" } else { "0" },
         result0.is_deterministic);
println!("  Qubit 1: {} (deterministic: {})", 
         if result1.outcome { "1" } else { "0" },
         result1.is_deterministic);

// Verify Bell state correlation: both qubits should always measure the same
assert_eq!(result0.outcome, result1.outcome, 
           "Bell state measurements must be correlated!");
println!("\nSuccess! Measurements are correlated as expected for a Bell state.");

Bell State Measurement Results:
  Qubit 0: 1 (deterministic: false)
  Qubit 1: 1 (deterministic: true)



In [97]:
// Symbolic Bell State Example using SymbolicSparseStab
// This simulator tracks measurement dependencies instead of collapsing to concrete outcomes

let mut sym_sim = SymbolicSparseStab::new(2);

// Create Bell state: (|00⟩ + |11⟩)/√2
sym_sim.h(0).cx(0, 1);

// Measure both qubits
let r0: SymbolicMeasurementResult = sym_sim.mz(0);
let r1: SymbolicMeasurementResult = sym_sim.mz(1);

println!("Symbolic Bell State Measurement Results:");
println!("  Measurement 0: {} (deterministic: {})", r0, r0.is_deterministic);
println!("  Measurement 1: {} (deterministic: {})", r1, r1.is_deterministic);

// The first measurement is non-deterministic (creates measurement index 0)
// The second measurement is deterministic and depends on measurement 0
println!("\nAnalysis:");
if !r0.is_deterministic {
    println!("  - Qubit 0 measurement was non-deterministic (random outcome)");
}
if r1.is_deterministic && r0.outcome == r1.outcome {
    println!("  - Qubit 1 measurement is deterministic and depends on measurement 0");
    println!("  - This shows the Bell state correlation: both qubits always measure the same!");
}

Success! Measurements are correlated as expected for a Bell state.
Symbolic Bell State Measurement Results:
  Measurement 0: m0=? (deterministic: false)
  Measurement 1: m1=m0 (deterministic: true)

Analysis:
  - Qubit 0 measurement was non-deterministic (random outcome)
  - Qubit 1 measurement is deterministic and depends on measurement 0
  - This shows the Bell state correlation: both qubits always measure the same!


()

In [115]:
// 3-qubit Repetition Code in Logical |+_L⟩ with Syndrome Measurements
//
// Qubits:
//   q0, q1, q2: Data qubits 
//   q3: Ancilla for Z0Z1 check
//   q4: Ancilla for Z1Z2 check
//
// Logical states:
//   |0_L⟩ = |000⟩
//   |1_L⟩ = |111⟩
//   |+_L⟩ = (|000⟩ + |111⟩)/√2
//
// Encoding circuit for |+_L⟩:
//   - Start with |+⟩|00⟩ (H on q0)
//   - CX(0,1), CX(0,2) to spread → (|000⟩ + |111⟩)/√2
//
// Syndrome extraction:
//   - Measure Z0Z1 using ancilla q3
//   - Measure Z1Z2 using ancilla q4
//
// Then measure data qubits in Z basis

let mut sim = SymbolicSparseStab::new(5);

println!("=== 3-Qubit Repetition Code: Logical |+_L⟩ ===\n");

// Encode logical |+_L⟩ = (|000⟩ + |111⟩)/√2
sim.h(0);           // |+⟩|00⟩
sim.cx(0, 1);       // (|00⟩ + |11⟩)|0⟩
sim.cx(0, 2);       // |000⟩ + |111⟩
println!("After encoding |+_L⟩ = (|000⟩ + |111⟩)/√2:");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Syndrome measurement: Z0Z1 via ancilla q3
// Circuit: H(q3), CX(q0,q3), CX(q1,q3), H(q3), Measure(q3)
sim.h(3);
sim.cx(0, 3);
sim.cx(1, 3);
sim.h(3);
println!("After Z0Z1 syndrome circuit (before measurement):");
println!("Stabilizers:\n{}", sim.stab_tableau());

let s0: SymbolicMeasurementResult = sim.mz(3);
println!("Syndrome S0 (Z0Z1) = {}, det={}", s0, s0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

// Syndrome measurement: Z1Z2 via ancilla q4
sim.h(4);
sim.cx(1, 4);
sim.cx(2, 4);
sim.h(4);
println!("After Z1Z2 syndrome circuit (before measurement):");
println!("Stabilizers:\n{}", sim.stab_tableau());

let s1: SymbolicMeasurementResult = sim.mz(4);
println!("Syndrome S1 (Z1Z2) = {}, det={}", s1, s1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

// Now measure data qubits in Z basis
let d0: SymbolicMeasurementResult = sim.mz(0);
println!("D0 (q0) = {}, det={}", d0, d0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let d1: SymbolicMeasurementResult = sim.mz(1);
println!("D1 (q1) = {}, det={}", d1, d1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let d2: SymbolicMeasurementResult = sim.mz(2);
println!("D2 (q2) = {}, det={}", d2, d2.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

println!("\n=== Measurement History ===");
println!("Total measurements: {}", sim.measurement_history().len());
println!("Deterministic: {}", sim.measurement_history().deterministic().len());
println!("Non-deterministic: {}", sim.measurement_history().nondeterministic().len());

// Use the new formatting convenience methods
println!("\nAll measurements:       {}", sim.measurement_history());
println!("Deterministic only:     {}", sim.measurement_history().format_deterministic());
println!("Non-deterministic only: {}", sim.measurement_history().format_nondeterministic());

// Show Debug format for one result (wrap in block to avoid lifetime issue)
println!("\nDebug format for first measurement:");
{
    let first = &sim.measurement_history()[0];
    println!("  {:?}", first);
}

println!("\n=== Summary ===");
println!("Measurement indices: S0=0, S1=1, D0=2, D1=3, D2=4");
println!("\nSyndromes should be deterministic {{}} (no errors in code space).");
println!("Data qubits: D0 is random, D1 and D2 are determined by D0 and syndromes.");
println!("\nDetectors: {{S0=0, S1=0, D1^D0=0, D2^D0=0}}")

=== 3-Qubit Repetition Code: Logical |+_L⟩ ===

After encoding |+_L⟩ = (|000⟩ + |111⟩)/√2:
Stabilizers:
{} ^ 0 XXXII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

After Z0Z1 syndrome circuit (before measurement):
Stabilizers:
{} ^ 0 XXXII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

Syndrome S0 (Z0Z1) = m0=0, det=true
Stabilizers:
{} ^ 0 XXXII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

After Z1Z2 syndrome circuit (before measurement):
Stabilizers:
{} ^ 0 XXXII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

Syndrome S1 (Z1Z2) = m1=0, det=true
Stabilizers:
{} ^ 0 XXXII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

D0 (q0) = m2=?, det=false
Stabilizers:
{2} ^ 0 ZIIII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

D1 (q1) = m3=m2, det=true
Stabilizers:
{2} ^ 0 ZIIII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

D2 (q2) = m4=m2, det=true
Stabilizers:
{2} ^ 0 ZIIII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ


=== Measurement History

()

In [118]:
// Sample from the measurement history using MeasurementSampler
//
// The MeasurementSampler efficiently generates many samples from the symbolic
// measurement history. It handles:
// - Random measurements (50/50 coin flip)
// - Deterministic fixed values (0 or 1)
// - Computed measurements (XOR of dependencies, optionally flipped)

use pecos::qsim::measurement_sampler::{MeasurementSampler, SampleResult};

// Create a sampler from the measurement history
let sampler: MeasurementSampler = MeasurementSampler::new(sim.measurement_history());

// Generate 1000 samples
let num_shots: usize = 1_000_000;
let result: SampleResult = sampler.sample(num_shots);

println!("=== Sampling from Repetition Code Measurement History ===\n");
println!("Measurement history: {}", sim.measurement_history());
println!("  m0/S0: Z0Z1 syndrome");
println!("  m1/S1: Z1Z2 syndrome");  
println!("  m2/D0: Data qubit 0 (random)");
println!("  m3/D1: Data qubit 1 (= m2)");
println!("  m4/D2: Data qubit 2 (= m2)");
println!("\nGenerated {} shots\n", num_shots);

// Display first 5 experiments in a nice table format
// Note: result.get() returns Bit which displays as 0/1
println!("First 5 experiments:");
println!("┌───────┬───────┬───────┬───────┬───────┬───────┐");
println!("│ Shot  │ m0/S0 │ m1/S1 │ m2/D0 │ m3/D1 │ m4/D2 │");
println!("├───────┼───────┼───────┼───────┼───────┼───────┤");
for shot in 0..5usize {
    let s0 = result.get(shot, 0);
    let s1 = result.get(shot, 1);
    let d0 = result.get(shot, 2);
    let d1 = result.get(shot, 3);
    let d2 = result.get(shot, 4);
    println!("│  {:3}  │   {}   │   {}   │   {}   │   {}   │   {}   │", shot, s0, s1, d0, d1, d2);
}
println!("└───────┴───────┴───────┴───────┴───────┴───────┘");

// Verify the expected correlations
println!("\n=== Verification ===");

// Count syndrome values - should all be 0 (no errors)
let s0_ones: usize = result.count_ones(0);
let s1_ones: usize = result.count_ones(1);
println!("m0/S0 (Z0Z1 syndrome): {} zeros, {} ones", num_shots - s0_ones, s0_ones);
println!("m1/S1 (Z1Z2 syndrome): {} zeros, {} ones", num_shots - s1_ones, s1_ones);

// Data qubits should be correlated (D0 = D1 = D2)
let d0_ones: usize = result.count_ones(2);
println!("\nm2/D0 distribution: {} zeros, {} ones (should be ~50/50)", num_shots - d0_ones, d0_ones);

// Check correlations
let mut d0_eq_d1: usize = 0;
let mut d0_eq_d2: usize = 0;
for shot in 0..num_shots {
    if result.get(shot, 2) == result.get(shot, 3) { d0_eq_d1 += 1; }
    if result.get(shot, 2) == result.get(shot, 4) { d0_eq_d2 += 1; }
}
println!("m2/D0 == m3/D1: {}/{} shots (should be 100%)", d0_eq_d1, num_shots);
println!("m2/D0 == m4/D2: {}/{} shots (should be 100%)", d0_eq_d2, num_shots);

=== Sampling from Repetition Code Measurement History ===

Measurement history: [m0=0, m1=0, m2=?, m3=m2, m4=m2]
  m0/S0: Z0Z1 syndrome
  m1/S1: Z1Z2 syndrome
  m2/D0: Data qubit 0 (random)
  m3/D1: Data qubit 1 (= m2)
  m4/D2: Data qubit 2 (= m2)

Generated 1000000 shots

First 5 experiments:
┌───────┬───────┬───────┬───────┬───────┬───────┐
│ Shot  │ m0/S0 │ m1/S1 │ m2/D0 │ m3/D1 │ m4/D2 │
├───────┼───────┼───────┼───────┼───────┼───────┤
│    0  │   0   │   0   │   1   │   1   │   1   │
│    1  │   0   │   0   │   0   │   0   │   0   │
│    2  │   0   │   0   │   0   │   0   │   0   │
│    3  │   0   │   0   │   1   │   1   │   1   │
│    4  │   0   │   0   │   1   │   1   │   1   │
└───────┴───────┴───────┴───────┴───────┴───────┘

=== Verification ===
m0/S0 (Z0Z1 syndrome): 1000000 zeros, 0 ones
m1/S1 (Z1Z2 syndrome): 1000000 zeros, 0 ones

m2/D0 distribution: 499329 zeros, 500671 ones (should be ~50/50)
m2/D0 == m3/D1: 1000000/1000000 shots (should be 100%)
m2/D0 == m4/D2: 100000

In [119]:
// Accessing individual shots and measurements
//
// Data is stored internally as packed u64 words (64 shots per word).
// Access methods extract individual bits:
//
// 1. result.get(shot, measurement) -> Bit
// 2. result.shot(shot) -> Bits  (all measurements for one shot)
// 3. result.format_shot(shot) -> String  (binary string like "01101")
// 4. result[(shot, measurement)] -> Bit (index syntax)
//
// The Bit type displays as 0/1 (not true/false) and supports all
// bitwise operations (^, &, |, !) while behaving like bool.
//
// The Bits type displays in standard binary format (LSB on right):
//   bits[0] appears on the RIGHT, bits[n-1] on the LEFT
//   Use bits.format_lsb_left() for array order (index 0 on left)

use pecos::prelude::Bits;

println!("=== Accessing Individual Shots ===\n");

// Get a single shot's results as Bits
let shot_0: Bits = result.shot(0);
println!("Shot 0 as Bits: {}", shot_0);  // LSB (m0) on right
println!("Shot 0 (LSB left): {}", shot_0.format_lsb_left());  // Index order
println!("  Length: {} measurements", shot_0.len());
println!("  Ones: {}, Zeros: {}", shot_0.count_ones(), shot_0.count_zeros());
println!("  Parity: {}", shot_0.parity());

// Access by measurement index
println!("\nAccessing shot 0 by measurement index:");
println!("  shot_0[0] (m0/S0) = {}", shot_0[0]);
println!("  shot_0[1] (m1/S1) = {}", shot_0[1]);
println!("  shot_0[2] (m2/D0) = {}", shot_0[2]);
println!("  shot_0[3] (m3/D1) = {}", shot_0[3]);
println!("  shot_0[4] (m4/D2) = {}", shot_0[4]);

// Alternative: use result.get(shot, measurement) for a single Bit
println!("\nUsing result.get(shot, measurement) -> Bit:");
println!("  result.get(0, 2) = {} (m2/D0 for shot 0)", result.get(0, 2));
println!("  result.get(1, 2) = {} (m2/D0 for shot 1)", result.get(1, 2));
println!("  result.get(2, 2) = {} (m2/D0 for shot 2)", result.get(2, 2));

// Pretty print first few shots
// Note: Bits displays with LSB on right (standard binary)
// So m4 appears leftmost, m0 appears rightmost
println!("\n=== First 10 Shots ===");
println!("       D2D1D0 S1S0  (standard binary: LSB on right)");
for i in 0..10usize {
    println!("Shot {:2}: {}", i, result.shot(i));
}

// Verify correlation for a shot using Bits methods
println!("\n=== Verify Correlations ===");
let shot_idx: usize = 42;
let bits: Bits = result.shot(shot_idx);
println!("Shot {}: {} (LSB right) = {} (LSB left)", 
         shot_idx, bits, bits.format_lsb_left());
// Bit supports == comparison directly
if bits[2] == bits[3] && bits[3] == bits[4] {
    println!("  D0=D1=D2 (as expected for repetition code)");
}

// Demonstrate parity check with Bits
println!("\n=== Parity Check ===");
// For repetition code, data qubits should have same parity pattern
let data_bits: Bits = Bits::new(vec![bits[2], bits[3], bits[4]]);
println!("Data bits (D0,D1,D2): {} (binary) = {} (index order)", 
         data_bits, data_bits.format_lsb_left());
println!("Parity of data bits: {}", data_bits.parity());

=== Accessing Individual Shots ===

Shot 0 as Bits: 11100
Shot 0 (LSB left): 00111
  Length: 5 measurements
  Ones: 3, Zeros: 2
  Parity: 1

Accessing shot 0 by measurement index:
  shot_0[0] (m0/S0) = 0
  shot_0[1] (m1/S1) = 0
  shot_0[2] (m2/D0) = 1
  shot_0[3] (m3/D1) = 1
  shot_0[4] (m4/D2) = 1

Using result.get(shot, measurement) -> Bit:
  result.get(0, 2) = 1 (m2/D0 for shot 0)
  result.get(1, 2) = 0 (m2/D0 for shot 1)
  result.get(2, 2) = 0 (m2/D0 for shot 2)

=== First 10 Shots ===
       D2D1D0 S1S0  (standard binary: LSB on right)
Shot  0: 11100
Shot  1: 00000
Shot  2: 00000
Shot  3: 11100
Shot  4: 11100
Shot  5: 00000
Shot  6: 11100
Shot  7: 00000
Shot  8: 00000
Shot  9: 00000

=== Verify Correlations ===
Shot 42: 00000 (LSB right) = 00000 (LSB left)
  D0=D1=D2 (as expected for repetition code)

=== Parity Check ===
Data bits (D0,D1,D2): 000 (binary) = 000 (index order)
Parity of data bits: 0


In [120]:
// More complex example: 3-qubit circuit with interesting measurement dependencies
// 
// Circuit:
//   q0: --H--@-------M0
//            |
//   q1: -----X--H--@-M1
//                  |
//   q2: -----------X-M2
//
// This creates a chain where:
// - q0 and q1 become entangled (Bell pair)
// - Then q1 and q2 become entangled
// - Measuring in this order should show interesting dependencies

let mut sim = SymbolicSparseStab::new(3);

// Create the entanglement chain
sim.h(0);        // q0 in superposition
sim.cx(0, 1);    // Entangle q0-q1
sim.h(1);        // q1 in superposition (relative to q0)
sim.cx(1, 2);    // Entangle q1-q2

// Measure all qubits
let m0: SymbolicMeasurementResult = sim.mz(0);
let m1: SymbolicMeasurementResult = sim.mz(1);
let m2: SymbolicMeasurementResult = sim.mz(2);

println!("3-Qubit Chain Measurement Results:");
println!("  M0 (qubit 0): {} (deterministic: {})", m0, m0.is_deterministic);
println!("  M1 (qubit 1): {} (deterministic: {})", m1, m1.is_deterministic);
println!("  M2 (qubit 2): {} (deterministic: {})", m2, m2.is_deterministic);

println!("\nInterpretation:");
println!("  - M0 outcome is random: {}", m0);
println!("  - M1 outcome is random: {}", m1);
println!("  - M2 outcome depends on: {}", m2);

// Show what XOR means
if m2.outcome.len() == 2 {
    println!("\n  M2 = {{0, 1}} means: M2_outcome = M0_outcome XOR M1_outcome");
}

3-Qubit Chain Measurement Results:
  M0 (qubit 0): m0=? (deterministic: false)
  M1 (qubit 1): m1=? (deterministic: false)
  M2 (qubit 2): m2=m1 (deterministic: true)

Interpretation:
  - M0 outcome is random: m0=?
  - M1 outcome is random: m1=?
  - M2 outcome depends on: m2=m1


()

In [121]:
// Trace through the 3-qubit circuit step by step with tableau display
// Note: The tableau format is "{measurement_indices} ^ flip PauliString"
//   - {} ^ 0: identity sign, no flip (deterministic 0)
//   - {} ^ 1: identity sign, flipped (deterministic 1)
//   - {0} ^ 0: depends on measurement 0, no flip
//   - {0,1} ^ 1: XOR of measurements 0,1, flipped
//
// Measurement results use Display format: m{index}^m{dep1}^m{dep2}^...={flip}
//   - m0=0: measurement 0 is deterministic 0
//   - m1^m0=0: measurement 1 equals measurement 0
//   - m2^m1^m0=1: measurement 2 equals m1 XOR m0 XOR 1

let mut sim = SymbolicSparseStab::new(3);

println!("Initial state:");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(0);
println!("After H(0):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.cx(0, 1);
println!("After CX(0,1):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(1);
println!("After H(1):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.cx(1, 2);
println!("After CX(1,2):");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Now measure
let m0: SymbolicMeasurementResult = sim.mz(0);
println!("After measuring qubit 0 ({}, det={}):", m0, m0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m1: SymbolicMeasurementResult = sim.mz(1);
println!("After measuring qubit 1 ({}, det={}):", m1, m1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m2: SymbolicMeasurementResult = sim.mz(2);
println!("After measuring qubit 2 ({}, det={}):", m2, m2.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

Initial state:
Stabilizers:
{} ^ 0 ZII
{} ^ 0 IZI
{} ^ 0 IIZ

After H(0):
Stabilizers:
{} ^ 0 XII
{} ^ 0 IZI
{} ^ 0 IIZ

After CX(0,1):
Stabilizers:
{} ^ 0 XXI
{} ^ 0 ZZI
{} ^ 0 IIZ

After H(1):
Stabilizers:
{} ^ 0 XZI
{} ^ 0 ZXI
{} ^ 0 IIZ

After CX(1,2):
Stabilizers:
{} ^ 0 XZI
{} ^ 0 ZXX
{} ^ 0 IZZ

After measuring qubit 0 (m0=?, det=false):
Stabilizers:
{0} ^ 0 ZII
{} ^ 0 ZXX
{} ^ 0 IZZ

After measuring qubit 1 (m1=?, det=false):
Stabilizers:
{0} ^ 0 ZII
{1} ^ 0 IZI
{} ^ 0 IZZ

After measuring qubit 2 (m2=m1, det=true):
Stabilizers:
{0} ^ 0 ZII
{1} ^ 0 IZI
{} ^ 0 IZZ



In [122]:
// Example where M2 depends on BOTH M0 and M1 (XOR)
//
// Circuit: Create a 3-qubit GHZ-like state, then do local rotations
//
//   q0: --H--@--------M0
//            |
//   q1: -----X--@-----M1
//               |
//   q2: --------X--H--M2
//
// The key insight: after CX gates, we have correlations.
// The H on q2 at the end converts Z2 to X2, which should
// create an XOR dependency when measured in Z basis.

let mut sim = SymbolicSparseStab::new(3);

println!("=== Circuit where M2 = M0 XOR M1 ===\n");

sim.h(0);
sim.cx(0, 1);
sim.cx(1, 2);
println!("After H(0), CX(0,1), CX(1,2) - GHZ state:");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(2);
println!("After H(2):");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Now measure
let m0: SymbolicMeasurementResult = sim.mz(0);
println!("After M0 ({}, det={}):", m0, m0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m1: SymbolicMeasurementResult = sim.mz(1);
println!("After M1 ({}, det={}):", m1, m1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m2: SymbolicMeasurementResult = sim.mz(2);
println!("After M2 ({}, det={}):", m2, m2.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

println!("Summary:");
println!("  M0 = {}", m0);
println!("  M1 = {}", m1);
println!("  M2 = {}", m2);
if m2.outcome.len() == 2 && m2.outcome.contains(0) && m2.outcome.contains(1) {
    println!("\n  m2^m1^m0=0 means: M2_outcome = M0_outcome XOR M1_outcome");
}

=== Circuit where M2 = M0 XOR M1 ===

After H(0), CX(0,1), CX(1,2) - GHZ state:
Stabilizers:
{} ^ 0 XXX
{} ^ 0 ZZI
{} ^ 0 IZZ

After H(2):
Stabilizers:
{} ^ 0 XXZ
{} ^ 0 ZZI
{} ^ 0 IZX

After M0 (m0=?, det=false):
Stabilizers:
{0} ^ 0 ZII
{} ^ 0 ZZI
{} ^ 0 IZX

After M1 (m1=m0, det=true):
Stabilizers:
{0} ^ 0 ZII
{} ^ 0 ZZI
{} ^ 0 IZX

After M2 (m2=?, det=false):
Stabilizers:
{0} ^ 0 ZII
{} ^ 0 ZZI
{2} ^ 0 IIZ

Summary:
  M0 = m0=?
  M1 = m1=m0
  M2 = m2=?


()

In [123]:
// Example demonstrating the flip flag from unitary operations
//
// The flip flag tracks phase changes from unitary gates:
// - X gate on |0⟩: Z stabilizer becomes -Z, so flip=1
// - Measurement of |1⟩ returns m0=1 (deterministic 1)

let mut sim = SymbolicSparseStab::new(1);

println!("=== Demonstrating the flip flag ===\n");

println!("Initial state |0⟩:");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Apply X gate to flip |0⟩ to |1⟩
sim.x(0);
println!("After X gate (now |1⟩):");
println!("Stabilizers:\n{}", sim.stab_tableau());
println!("Notice: {{}} ^ 1 Z means the stabilizer is -Z (flip=1)\n");

// Measure - should be deterministic 1
let m: SymbolicMeasurementResult = sim.mz(0);
println!("Measurement result: {}", m);
println!("  - Empty dependency set means no measurement dependencies");
println!("  - =1 means the result is flipped, giving outcome 1");

// Reset and show double-X cancellation
sim.reset();
sim.x(0);
sim.x(0);
println!("\nAfter X·X (identity):");
println!("Stabilizers:\n{}", sim.stab_tableau());
println!("Notice: {{}} ^ 0 Z means we're back to +Z (flip=0)")

=== Demonstrating the flip flag ===

Initial state |0⟩:
Stabilizers:
{} ^ 0 Z

After X gate (now |1⟩):
Stabilizers:
{} ^ 1 Z

Notice: {} ^ 1 Z means the stabilizer is -Z (flip=1)

Measurement result: m0=1
  - Empty dependency set means no measurement dependencies
  - =1 means the result is flipped, giving outcome 1

After X·X (identity):
Stabilizers:
{} ^ 0 Z

Notice: {} ^ 0 Z means we're back to +Z (flip=0)


()

In [124]:
// Example combining measurement dependencies with flip flag
//
// Circuit: X on qubit 0, then create Bell state
//   q0: --X--H--@--M0
//               |
//   q1: --------X--M1
//
// This shows how the flip propagates through the circuit

let mut sim = SymbolicSparseStab::new(2);

println!("=== Combining measurement dependencies with flip ===\n");

sim.x(0);
println!("After X(0):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(0);
println!("After H(0):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.cx(0, 1);
println!("After CX(0,1) - Bell state with phase:");
println!("Stabilizers:\n{}", sim.stab_tableau());

let m0: SymbolicMeasurementResult = sim.mz(0);
let m1: SymbolicMeasurementResult = sim.mz(1);

println!("Measurement results:");
println!("  {}", m0);
println!("  {}", m1);
println!("\nBoth measurements depend on measurement index 0.");
println!("The flips indicate phase accumulated from the initial X gate.")

=== Combining measurement dependencies with flip ===

After X(0):
Stabilizers:
{} ^ 1 ZI
{} ^ 0 IZ

After H(0):
Stabilizers:
{} ^ 1 XI
{} ^ 0 IZ

After CX(0,1) - Bell state with phase:
Stabilizers:
{} ^ 1 XX
{} ^ 0 ZZ

Measurement results:
  m0=?
  m1=m0

Both measurements depend on measurement index 0.
The flips indicate phase accumulated from the initial X gate.


()

In [ ]:
// Maybe make {} ^ not printed and 1 -> - and 0 not printed... maybe non-empty {} should be like (-1)^{m0^m1^..}

## Python Integration: Guppy → HUGR → Symbolic Execution

The symbolic stabilizer simulator is also available from Python via the `pecos.experimental` module. This enables a powerful workflow:

1. **Write quantum circuits in Guppy** (Python-embedded DSL)
2. **Compile to HUGR** (Hierarchical Unified Graph Representation)
3. **Execute symbolically** to get measurement dependencies
4. **Sample efficiently** - millions of shots in milliseconds!

### Example: Bell State from Guppy

```python
from guppylang import guppy
from guppylang.std.quantum import h, cx, measure, qubit
from pecos.experimental import execute_hugr_symbolic

# Define a Bell state circuit in Guppy
@guppy
def bell_circuit() -> tuple[bool, bool]:
    q0 = qubit()
    q1 = qubit()
    h(q0)
    cx(q0, q1)
    return (measure(q0), measure(q1))

# Compile to HUGR and execute symbolically
package = bell_circuit.compile()
result = execute_hugr_symbolic(package.to_bytes())

# Inspect the symbolic measurement dependencies
print(f"Result: {result}")
# Output: SymbolicExecutionResult(measurements=2, deterministic=1, random=1)

print(f"Symbolic structure: {result}")
# Output: [m0=?, m1=m0]
# This shows: m0 is random, m1 always equals m0 (Bell correlation!)

# Generate 1 million samples - extremely fast!
samples = result.sample(1_000_000)

# Or get counts directly
counts = result.sample_counts(1_000_000)
print(counts)
# Output: {b'\x00\x00': ~500000, b'\x01\x01': ~500000}
```

### Why is sampling so fast?

The symbolic simulator tracks measurement dependencies using XOR relationships:
- Each measurement is either **deterministic** (fixed value or depends on previous measurements) or **random** (50/50 coin flip)
- Sampling is reduced to generating random bits and computing XORs - no quantum simulation needed!
- This makes generating millions of samples nearly instantaneous

### Available Functions

```python
from pecos.experimental import (
    execute_hugr_symbolic,      # Execute HUGR bytes symbolically
    execute_dag_circuit_symbolic,  # Execute DagCircuit directly
    SymbolicExecutionResult,    # Result type with sampling methods
)
```

### SymbolicExecutionResult Methods

- `num_measurements` - Total number of measurements
- `num_deterministic` - Measurements with fixed or computed outcomes
- `num_nondeterministic` - Random measurements (coin flips)
- `sample(num_shots)` - Generate samples as list of bool lists
- `sample_counts(num_shots)` - Generate counts as dict